In [ ]:
# @title Environment setup
%set_env UV_PROJECT_ENVIRONMENT /usr/local
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jane@doe.eu"
!git config --global user.name "Jane Doe"
!git config --global init.defaultBranch main
!git clone --branch solution https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv sync --all-extras

# MLflow, model registry & deployment

This demonstration walks through the solution of the lab of the same name, on the `solution` branch of the [`landscape-classifier` repository](https://github.com/shuuchuu/landscape-classifier/tree/solution): publishing a model to the MLflow model registry, using this registry, then serving the model with a FastAPI API packaged with Docker.

## MLflow credentials

Enter your DagsHub credentials (the password is the secret token, not your DagsHub password):

In [ ]:
username = ""
token = ""

In [ ]:
# @title To run
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Data retrieval

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

## Publishing a model to a model registry

The training script tracks metrics with [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) (`autolog`), as well as the training dataset. It then wraps the Keras model in a custom `pyfunc` model (`WrappedModel`) that embeds the image preprocessing, publishes it to the model registry as `dev.ml.landscape-classifier`, and gives the new version the `champion` alias.

In [ ]:
!cat src/landscape_classifier/train.py

Its parameters are managed by DVC, in `params.yaml`:

In [ ]:
!cat params.yaml

In [ ]:
!landscape-classifier

## Using the MLflow model registry

### Creating an MLflow client

The tracking address is correctly read from the environment variable defined at the top of this notebook:

In [ ]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

### Model search

All the registered models, each with its most recent version:

In [ ]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

### Retrieving a model from the model registry

An MLflow model can have several *flavors*, as many ways to load it. Ours was published as a custom `pyfunc` model: its only flavor is `python_function`, which gives the whole pipeline (image preprocessing, prediction, result formatting). A Keras flavor would only have given the network, without the preprocessing.

In [ ]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

### Adding an alias & tags

Aliases and tags are a way to organize models in an MLflow model registry. Aliases always target a specific version of a model while tags can also apply to all versions of a model.

The training script already gave the `champion` alias to the last version. We add a `domain` tag with value `cv` to the model and a `size` tag with value `small` to its last version:

In [ ]:
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")
client.get_model_version_by_alias(MODEL_NAME, "champion")

We can then retrieve the model by its alias rather than its exact version:

In [ ]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

### Changing model phases

To promote a model in the production pipeline, the recommended way is to copy the chosen version to a new model name, here the one of the staging environment:

In [ ]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

## API creation with FastAPI

The preprocessing is the `process_image` function, used both for training and for inference (in `WrappedModel`): this is the point that needs extreme care, since a different preprocessing between training and inference degrades predictions without raising any error.

In [ ]:
!cat src/landscape_classifier/data.py

The API loads the model only once, at startup, from the registry (by its alias), describes its return type with [Pydantic](https://docs.pydantic.dev/latest/) and accepts images [sent with POST](https://fastapi.tiangolo.com/tutorial/request-files/):

In [ ]:
!cat src/landscape_classifier/api.py

### Starting the API

We start the server in the background, then wait for it to answer (loading the model can take a while):

In [ ]:
!nohup uvicorn landscape_classifier.api:app --port 8000 > api.log 2>&1 &

In [ ]:
import time

import requests

for _ in range(300):
  try:
    requests.get("http://127.0.0.1:8000/docs", timeout=1)
    break
  except requests.ConnectionError:
    time.sleep(1)
print(Path("api.log").read_text())

### Calling the API

In [ ]:
with open("forest1.jpg", "rb") as forest1, open("forest2.jpg", "rb") as forest2:
  response = requests.post(
    "http://127.0.0.1:8000/",
    files=[("images", forest1), ("images", forest2)],
  )
response.raise_for_status()
response.json()

## Dockerfile creation

Docker isn't available in Colab: here is the Dockerfile of the solution. A first stage exports the dependencies with `uv`, the second one installs them on the Python 3.12 base image along with the project, then starts the FastAPI server.

In [ ]:
!cat Dockerfile

On a machine with Docker, we build then run the image, passing it the MLflow credentials as environment variables:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

The API documentation, which also serves as a test client, is then at <http://localhost:8000/docs>.

## API improvements

Several choices of the solution make the API suited to production deployment with a Docker image:

- the model is loaded only once, at startup, not on every request;
- the MLflow credentials are given when starting the container, never copied into the image;
- the API answers 503 if the model couldn't be loaded.

We could go further by making the model name and alias configurable with environment variables, to serve the same image in development and in staging.